In [1]:
from sklearn.datasets import fetch_20newsgroups
import pandas as pd

categories = [
    "comp.graphics",
    "rec.sport.baseball",
    "sci.space",
    "talk.politics.misc"
]

train_data = fetch_20newsgroups(
    subset = 'train',
    categories = categories,
    remove = ('headers','footers','quotes'),
    random_state = 42
)

test_data = fetch_20newsgroups(
    subset = 'train',
    categories = categories,
    remove = ('headers','footers','quotes'),
    random_state = 42
)

print(f'Lenght of Training and Test datasets: {len(train_data.data)}  {len(test_data.data)}')
print(f'Target Categories: {train_data.target_names} ')

Lenght of Training and Test datasets: 2239  2239
Target Categories: ['comp.graphics', 'rec.sport.baseball', 'sci.space', 'talk.politics.misc'] 


In [2]:
df = pd.DataFrame({
    "text": train_data.data,
    "label": train_data.target
})

In [3]:
df['label'].value_counts()

label
1    597
2    593
0    584
3    465
Name: count, dtype: int64

In [4]:
df.head()

,text,label
0,I thought that was Sandy Koufax.,1
1,"\nAnd the religious right worships engines, sm...",3
2,\nHow can a witness tell that someone in a bur...,3
3,"\n\n\n\nYes, long before Star Trek. Before Ei...",2
4,\n\nIt depends. If you can get your old veter...,1


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(),
    "Random Forest Regressor": RandomForestClassifier(n_estimators=100,random_state=42)
}

results = {}
trained_models = {}

for name, classifier in models.items():
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=50000,
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True
        )),
        ("classifier", classifier)
    ])

    pipeline.fit(train_data.data, train_data.target)

    predictions = pipeline.predict(test_data.data)

    results[name] = {
        "Accuracy": accuracy_score(test_data.target, predictions),
        "Macro F1": f1_score(
            test_data.target, predictions, average="macro"
        )
    }

    trained_models[name] = pipeline

results_df = pd.DataFrame(results).T.sort_values(
    "Macro F1", ascending=False
)

print(results_df)

                         Accuracy  Macro F1
Random Forest Regressor  0.979902  0.980485
Linear SVM               0.979902  0.980485
Logistic Regression      0.974989  0.975558
Naive Bayes              0.964716  0.965521


In [6]:
best_name = results_df.index[0]
best_model = trained_models[best_name]

predictions = best_model.predict(test_data.data)

print("Best model:", best_name)

print(classification_report(
    test_data.target,
    predictions,
    target_names=test_data.target_names,
    zero_division=0
))

Best model: Random Forest Regressor
                    precision    recall  f1-score   support

     comp.graphics       1.00      0.97      0.99       584
rec.sport.baseball       0.93      1.00      0.96       597
         sci.space       1.00      0.97      0.99       593
talk.politics.misc       1.00      0.97      0.99       465

          accuracy                           0.98      2239
         macro avg       0.98      0.98      0.98      2239
      weighted avg       0.98      0.98      0.98      2239



In [7]:
examples = [
    "NASA announced a new mission to explore Mars.",
    "The baseball team won the championship after a close game.",
    "The government debated a new national policy.",
    "Computer graphics can generate realistic three-dimensional scenes."
]

predictions = best_model.predict(examples)

for text, label in zip(examples, predictions):
    print(f"\nText: {text}")
    print(f"Predicted category: {train_data.target_names[label]}")


Text: NASA announced a new mission to explore Mars.
Predicted category: sci.space

Text: The baseball team won the championship after a close game.
Predicted category: rec.sport.baseball

Text: The government debated a new national policy.
Predicted category: rec.sport.baseball

Text: Computer graphics can generate realistic three-dimensional scenes.
Predicted category: comp.graphics
